## tl;dr
Bahrain, 1er octobre 2026 : six tours et arrivée pilote confirmés. Capture carburant cohérente avec DuckDB (<0,004 L d'écart maximal). Le HUD total peut descendre de ~6,5 à ~5,9 après zéro chrono, avant l'arrivée réelle au tour6. Aucune promotion ML; accidents/spins confirmés par le pilote aux tours5–6.


## Context & Methods
Audit diagnostic, aucune optimisation ni apprentissage. Sources en lecture seule; les fichiers bruts ne sont ni déplacés ni modifiés.
### Key Assumptions
Temps natifs de chaque canal conservés; interpolation linéaire du carburant20Hz aux événements de ligne. Le tour natif0 inclut formation et premier tour : le départ course est isolé via lap_start_s au vert. HUD retranscrit visuellement aux timestamps vidéo, synchronisation approximative26,4s, pas précision image par image. Valeur future d'arrivée utilisée seulement comme vérité rétrospective, jamais comme entrée d'une prédiction.


In [ ]:
from pathlib import Path
import importlib.util
root = Path.cwd()
if not (root / 'scripts').exists():
    root = root.parent
spec = importlib.util.spec_from_file_location('race_audit', root / 'scripts/audit_bahrain_native_race.py')
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
result = module.audit()


## Data
Sources et empreintes calculées pendant l'audit; capture initiale inachevée exclue, mais conservée. La date UTC du2octobre correspond au1eroctobre à Toronto.


In [ ]:
print(result['sources'])
assert result['capture_samples'] == 3899
assert result['final_completed_laps'] == 6
assert result['fuel_reconciliation']['max_abs_difference_l'] < 0.004
assert 19 < result['finish_after_leader_s'] < 21
assert 64 < result['finish_after_clock_zero_s'] < 66


## Results
Consommation descriptive par tour réel; les tours5–6 ne sont pas des références push propres. Le premier tour ne doit pas absorber la consommation de formation.


In [ ]:
for lap in result['boundaries']:
    print(lap)
print('Fuel reconciliation:', result['fuel_reconciliation'])
print('Formation litres:', result['formation_fuel_l'])
print('Race litres:', result['race_fuel_l'])
print('Finish fuel:', result['fuel_at_finish_l'])
assert abs(result['formation_fuel_l'] + result['race_fuel_l'] + result['fuel_at_finish_l'] - 55) < 1e-8


## Takeaways
La collecte est exploitable pour développer l'adaptateur de contexte course. Ne pas arrêter au chrono zéro, à la phase8 ou à l'arrivée du meneur. Séparer resets garage et formation; conserver le restant du tour tant que le pilote n'a pas fini. Ne pas convertir mécaniquement le HUD fractionnaire. Une course avec beaucoup de carburant ne valide ni économie LICO, ni marge anti-panne, ni arrêts, ni tours de retard. Le relevé vidéo247–249s montre aussi que l'autonomie HUD se réajuste après la ligne.
